# Macroeconomics Assignment — Chunk 1: Data Preparation, Cleaning, and Validation

**Course:** Macroeconomics  
**Topic:** Indian Household Economics (Income, Expenditure, and EMI Dynamics: 2014–15 to 2025–26)  
**Objective:** Build a robust, reproducible data extraction, cleaning, and validation pipeline across all 36 annual dataset files (12 years × 3 categories), handle non-standard tab-separated text formats, isolate national aggregates, audit missing data patterns, analyze source footnotes, and save verified analysis-ready datasets for Chunk 2.

## 1. Project Environment & Root Directory Discovery

We locate the project root dynamically relative to the working directory without hardcoding machine-specific absolute paths.

In [1]:
from pathlib import Path
import sys

# Locate project root dynamically
current_dir = Path.cwd()
candidates = [current_dir, *current_dir.resolve().parents]

PROJECT_ROOT = next(
    (
        p for p in candidates
        if (p / "data" / "raw").is_dir() and (p / "notebooks").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed output directory: {OUTPUT_DIR}")

Project root: C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics
Raw data directory: C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\data\raw
Processed output directory: C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\outputs\processed


## 2. Raw File Audit & Signature Inspection

The 36 files have `.xls` extensions but are actually tab-separated text files exported from the data portal. We verify file signatures and confirm that no files are binary OLE workbooks (`.xls`) or ZIP-based archives (`.xlsx`).

In [2]:
categories = ["Income", "Expenditure", "EMI"]
file_audit = []

for cat in categories:
    cat_folder = RAW_DIR / cat
    files = sorted(cat_folder.glob("*.xls"))
    for f in files:
        with open(f, "rb") as fp:
            sig = fp.read(8)
        
        if sig.startswith(b"\xD0\xCF\x11\xE0"):
            fmt = "Binary Excel (OLE)"
        elif sig.startswith(b"PK"):
            fmt = "ZIP archive (XLSX)"
        else:
            fmt = "Tab-separated text (TSV)"
            
        file_audit.append({
            "Category": cat,
            "Filename": f.name,
            "SizeBytes": f.stat().st_size,
            "DetectedFormat": fmt
        })

import pandas as pd
audit_files_df = pd.DataFrame(file_audit)
print(f"Total raw files audited: {len(audit_files_df)}")
print("Format distribution:")
print(audit_files_df["DetectedFormat"].value_counts())
display(audit_files_df.groupby("Category").agg(FileCount=("Filename", "count"), MinSize=("SizeBytes", "min"), MaxSize=("SizeBytes", "max")))

Total raw files audited: 36
Format distribution:
DetectedFormat
Tab-separated text (TSV)    36
Name: count, dtype: int64


,FileCount,MinSize,MaxSize
Category,,,
EMI,12,3740,4540
Expenditure,12,3401,3795
Income,12,8627,11355


## 3. Data Cleaning and Validation Pipeline Execution

We use the modular cleaning engine implemented in `src/data_cleaner.py` to parse, clean, and validate all 36 files.

### Key Technical Features of the Pipeline:
1. **Dynamic Header & Metadata Handling:**
   - Title (Row 0) and Subtitle/Unit (Row 1) are preserved in metadata.
   - Area and Measure headers are combined into standardized identifiers (`Total_Wages`, `Urban_House_EMI`, `Total_Households_million`, etc.).
   - Quotation marks (present in historical files like `EMI_2014_15.xls`) and thousands commas are stripped.
2. **Missing Data Policy:**
   - Blank cells are strictly preserved as `NaN` (never imputed with zero).
3. **Isolation of National Aggregate:**
   - The `India` aggregate row is separated from state/UT observations into dedicated national time-series datasets.
4. **Sampling & Footnote Capture:**
   - Trailing disclaimers (e.g. the March 2026 75% sample footnote) are parsed into audit records.

In [3]:
from src.data_cleaner import process_all_files, save_processed_outputs

# Execute the pipeline across all 36 files
cleaned_states, national_data, audit_df, source_notes_df, state_consistency_df = process_all_files(RAW_DIR)

# Save all processed datasets and verification text
summary_report = save_processed_outputs(
    cleaned_states=cleaned_states,
    national_data=national_data,
    audit_df=audit_df,
    source_notes_df=source_notes_df,
    state_consistency_df=state_consistency_df,
    output_dir=OUTPUT_DIR
)

C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\src\data_cleaner.py:119: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace('', np.nan)
C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\src\data_cleaner.py:119: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace('', np.nan)
C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\src\data_cleaner.py:119: FutureWarning: Downcasting behavior in `replace` is deprecated and

C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\src\data_cleaner.py:119: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace('', np.nan)
C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\src\data_cleaner.py:119: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace('', np.nan)
C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\src\data_cleaner.py:119: FutureWarning: Downcasting behavior in `replace` is deprecated and

Successfully saved all 9 CSV outputs and chunk1_verification.txt to C:\Users\harsh\OneDrive\HARSHIT_HOME\VIT\SEMESTER 5\Macroeconomics\household-economics\outputs\processed


## 4. Data Quality Audit & Invariant Checks

We examine the per-file data quality audit metrics:
- Check that each file contains exactly 28 State/UT records and 1 India aggregate record.
- Verify that duplicate state records are strictly 0.
- Inspect missing numeric cell counts per file.

In [4]:
print("=== DATA QUALITY AUDIT TABLE (Summary by Category) ===")
cat_summary = audit_df.groupby("Category").agg(
    TotalFiles=("File", "count"),
    TotalStateRows=("State_UT_rows", "sum"),
    TotalIndiaRows=("India_rows", "sum"),
    TotalMissingCells=("Missing_numeric_cells", "sum"),
    TotalDuplicateStates=("Duplicate_state_rows", "sum"),
    UnexpectedIndiaRows=("Unexpected_india_rows", "sum")
)
display(cat_summary)

print("\n=== FULL AUDIT PER YEAR AND CATEGORY ===")
display(audit_df.pivot(index="Year", columns="Category", values=["State_UT_rows", "Missing_numeric_cells"]))

=== DATA QUALITY AUDIT TABLE (Summary by Category) ===


,TotalFiles,TotalStateRows,TotalIndiaRows,TotalMissingCells,TotalDuplicateStates,UnexpectedIndiaRows
Category,,,,,,
EMI,12,336,12,270,0,0
Expenditure,12,336,12,216,0,0
Income,12,336,12,1736,0,0



=== FULL AUDIT PER YEAR AND CATEGORY ===


State_UT_rows                    Missing_numeric_cells              \
Category           EMI Expenditure Income                   EMI Expenditure   
Year                                                                          
2014-15             28          28     28                    55          44   
2015-16             28          28     28                    55          44   
2016-17             28          28     28                    55          44   
2017-18             28          28     28                    25          20   
2018-19             28          28     28                    10           8   
2019-20             28          28     28                    10           8   
2020-21             28          28     28                    10           8   
2021-22             28          28     28                    10           8   
2022-23             28          28     28                    10           8   
2023-24             28          28     28                    10           8   
2024-25             28          28     28                    10           8   
2025-26             28          28     28                    10           8   

                 
Category Income  
Year             
2014-15     278  
2015-16     278  
2016-17     278  
2017-18     218  
2018-19     188  
2019-20     188  
2020-21     188  
2021-22      24  
2022-23      24  
2023-24      24  
2024-25      24  
2025-26      24

## 5. State & Union Territory Label Consistency Audit

We verify geographic consistency across all 12 years and across Income, Expenditure, and EMI categories.

In [5]:
display(state_consistency_df)

# Confirm state list
unique_states = sorted(cleaned_states["Income"]["State"].unique())
print(f"Total unique states/UTs: {len(unique_states)}")
for idx, st in enumerate(unique_states, start=1):
    print(f"{idx:2d}. {st}")

,Category,Total_Years,Total_Records,Unique_state_labels,Missing_vs_Income,Extra_vs_Income,Consistent_With_All
0,Income,12,336,28,None,None,True
1,Expenditure,12,336,28,None,None,True
2,EMI,12,336,28,None,None,True


Total unique states/UTs: 28
 1. Andhra Pradesh
 2. Assam
 3. Bihar
 4. Chandigarh
 5. Chhattisgarh
 6. Delhi
 7. Goa
 8. Gujarat
 9. Haryana
10. Himachal Pradesh
11. Jammu & Kashmir
12. Jharkhand
13. Karnataka
14. Kerala
15. Madhya Pradesh
16. Maharashtra
17. Meghalaya
18. Odisha
19. Puducherry
20. Punjab
21. Rajasthan
22. Sikkim
23. Tamil Nadu
24. Telangana
25. Tripura
26. Uttar Pradesh
27. Uttarakhand
28. West Bengal


## 6. Source Notes & Sampling Disclaimer Analysis

We examine the source notes detected across files. We check why the note `'2025-2026 observation is based on 75% sample for Mar 2026.'` appears across multiple files and document its economic meaning.

In [6]:
display(source_notes_df[["Year", "Category", "File", "Subtitle_Unit", "Note_Classification", "Applies_To_Current_File"]].head(15))
notes_with_disclaimer = source_notes_df["Detected_Note"].str.contains("75% sample", na=False).sum()
print("Total files with 75% sample footnote:", notes_with_disclaimer, "of 36")

,Year,Category,File,Subtitle_Unit,Note_Classification,Applies_To_Current_File
0,2014-15,EMI,EMI_2014_15.xls,India : Rs. million : 2014-15,Global Portal Export Disclaimer,False
1,2015-16,EMI,EMI_2015_16.xls,India : Rs. million : 2015-16,Global Portal Export Disclaimer,False
2,2016-17,EMI,EMI_2016_17.xls,India : Rs. million : 2016-17,Global Portal Export Disclaimer,False
3,2017-18,EMI,EMI_2017_18.xls,India : Rs. million : 2017-18,Global Portal Export Disclaimer,False
4,2018-19,EMI,EMI_2018_19.xls,India : Rs. million : 2018-19,Global Portal Export Disclaimer,False
5,2019-20,EMI,EMI_2019_20.xls,India : Rs. million : 2019-20,Global Portal Export Disclaimer,False
6,2020-21,EMI,EMI_2020_21.xls,India : Rs. million : 2020-21,Global Portal Export Disclaimer,False
7,2021-22,EMI,EMI_2021_22.xls,India : Rs. million : 2021-22,Global Portal Export Disclaimer,False
8,2022-23,EMI,EMI_2022_23.xls,India : Rs. million : 2022-23,Global Portal Export Disclaimer,False
9,2023-24,EMI,EMI_2023_24.xls,India : Rs. million : 2023-24,Global Portal Export Disclaimer,False


Total files with 75% sample footnote: 34 of 36


## 7. Processed Dataset Verification & Previews

We reload the saved processed datasets from disk and inspect their shapes, column names, and head observations.

In [7]:
files_to_check = [
    "income_states_all_years.csv",
    "expenditure_states_all_years.csv",
    "emi_states_all_years.csv",
    "income_national_all_years.csv",
    "expenditure_national_all_years.csv",
    "emi_national_all_years.csv",
    "data_quality_audit.csv",
    "source_notes.csv",
    "state_consistency.csv"
]

print("Verifying saved files on disk:")
for fname in files_to_check:
    fpath = OUTPUT_DIR / fname
    assert fpath.exists(), f"Missing output file: {fname}"
    reloaded = pd.read_csv(fpath)
    print(f" - {fname:<35} : Shape {reloaded.shape}")

print("\n=== INCOME STATE DATASET PREVIEW ===")
inc_df = pd.read_csv(OUTPUT_DIR / "income_states_all_years.csv")
display(inc_df.iloc[:5, :8])

print("\n=== EXPENDITURE STATE DATASET PREVIEW ===")
exp_df = pd.read_csv(OUTPUT_DIR / "expenditure_states_all_years.csv")
display(exp_df.iloc[:5, :8])

print("\n=== EMI STATE DATASET PREVIEW ===")
emi_df = pd.read_csv(OUTPUT_DIR / "emi_states_all_years.csv")
display(emi_df.iloc[:5, :8])

print("\n=== NATIONAL AGGREGATES PREVIEW (First 3 Years) ===")
inc_nat = pd.read_csv(OUTPUT_DIR / "income_national_all_years.csv")
display(inc_nat[["Year", "State", "Total_All sources of income", "Total_Wages", "Total_Profits"]].head(3))

Verifying saved files on disk:
 - income_states_all_years.csv         : Shape (336, 38)
 - expenditure_states_all_years.csv    : Shape (336, 14)
 - emi_states_all_years.csv            : Shape (336, 17)
 - income_national_all_years.csv       : Shape (12, 38)
 - expenditure_national_all_years.csv  : Shape (12, 14)
 - emi_national_all_years.csv          : Shape (12, 17)
 - data_quality_audit.csv              : Shape (36, 10)
 - source_notes.csv                    : Shape (36, 9)
 - state_consistency.csv               : Shape (3, 7)

=== INCOME STATE DATASET PREVIEW ===


,Year,State,Total_All sources of income,Total_Wages,Total_Pensions,Total_Imputed income,Total_Profits,Total_Interest and dividends
0,2014-15,Andhra Pradesh,1869686.70,1512549.25,46125.43,9839.52,268669.30,178.79
1,2014-15,Assam,970472.12,734760.00,63679.33,2628.14,127127.25,782.01
2,2014-15,Bihar,2373878.55,1847346.01,44600.36,12334.85,443853.40,31.59
3,2014-15,Chandigarh,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-15,Chhattisgarh,753519.57,646489.75,14979.69,33658.44,49653.17,420.68



=== EXPENDITURE STATE DATASET PREVIEW ===


,Year,State,Total_Households_million,Total_Total_expenditure_million_Rs,Total_Average_expenditure_Rs,Total_Median_expenditure_Rs,Urban_Households_million,Urban_Total_expenditure_million_Rs
0,2014-15,Andhra Pradesh,13.56,1134170.87,83618.0,79741.0,4.07,370622.29
1,2014-15,Assam,7.04,613764.96,87231.0,77618.0,1.12,137924.71
2,2014-15,Bihar,20.98,1577071.05,75176.0,69015.0,2.35,223112.50
3,2014-15,Chandigarh,NaN,NaN,NaN,NaN,0.26,38524.67
4,2014-15,Chhattisgarh,6.35,546635.80,86063.0,78008.0,1.49,160021.32



=== EMI STATE DATASET PREVIEW ===


,Year,State,Total_All_EMIs,Total_House_EMI,Total_Vehicle_EMI,Total_Durables_EMI,Total_Other_EMI,Urban_All_EMIs
0,2014-15,Andhra Pradesh,8699.78,1796.42,997.45,222.97,5682.93,2313.10
1,2014-15,Assam,23411.12,5746.90,7358.26,1018.64,9287.32,5469.69
2,2014-15,Bihar,0.65,0.00,0.65,0.00,0.00,0.65
3,2014-15,Chandigarh,NaN,NaN,NaN,NaN,NaN,163.87
4,2014-15,Chhattisgarh,4188.41,661.56,3075.39,64.36,387.10,760.95



=== NATIONAL AGGREGATES PREVIEW (First 3 Years) ===


,Year,State,Total_All sources of income,Total_Wages,Total_Profits
0,2014-15,India,45161310.38,33146914.97,8979391.18
1,2015-16,India,47338698.53,36020510.43,7733065.49
2,2016-17,India,51026119.57,40602068.37,6357592.97


## 8. Verification Summary & Invariant Assertions

We run strict automated assertions ensuring that:
1. All 12 financial years are present for every category.
2. Exactly 28 states/UTs exist for every year.
3. No unexpected national aggregate rows or duplicate state rows exist.
4. No numeric columns are entirely missing across all years.

In [8]:
# Invariant Assertions
expected_years = [
    "2014-15", "2015-16", "2016-17", "2017-18", "2018-19", "2019-20",
    "2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26"
]

for cat, df in cleaned_states.items():
    assert sorted(df["Year"].unique()) == expected_years, f"{cat} years mismatch"
    assert len(df) == 336, f"{cat} row count mismatch (expected 336, got {len(df)})"
    assert df["State"].nunique() == 28, f"{cat} state count mismatch (expected 28)"
    assert df.duplicated(subset=["Year", "State"]).sum() == 0, f"{cat} has duplicate year-state pairs"

for cat, df in national_data.items():
    assert sorted(df["Year"].unique()) == expected_years, f"{cat} national years mismatch"
    assert len(df) == 12, f"{cat} national row count mismatch (expected 12, got {len(df)})"

# Assert that no numeric column became entirely NaN across all years
for cat, df in cleaned_states.items():
    numeric_cols = [c for c in df.columns if c not in ["Year", "State"]]
    all_nan_cols = [c for c in numeric_cols if df[c].isna().all()]
    assert len(all_nan_cols) == 0, f"{cat} has entirely missing columns: {all_nan_cols}"

print("All invariant assertions PASSED with 100% compliance!")

All invariant assertions PASSED with 100% compliance!
